# Getting started with Apographe

Apographe collects public data, day by day, into plain CSV files. This notebook runs a built-in source, reads what it wrote, then collects from a source you write yourself.

In [ ]:
%pip install -q apographe pandas

## 1. Run a built-in source

`cy.ekalathi` is Cyprus's supermarket price site. `limit=5` collects 5 products, which takes about a minute (Apographe waits a few seconds between requests).

In [ ]:
from apographe import run

report = run("cy.ekalathi", out="data", limit=5)
report.rows  # rows written per table

## 2. Read the files

Everything is plain CSV, with ids in every row so the tables join.

In [ ]:
import glob
import pandas as pd

products = pd.read_csv("data/cy/ekalathi/products.csv")
prices = pd.concat(map(pd.read_csv, glob.glob("data/cy/ekalathi/store_prices/*/*.csv.gz")))
products.head()

In [ ]:
prices.head()

## 3. Fuel prices (needs Chrome)

`cy.fuel` collects every petrol station's price for each fuel. Its default job scrapes the Ministry's website, so it needs Google Chrome installed and the `cy-fuel` extras. It takes about a minute. (There is also an `api` job that uses the official API; it needs approval and credentials, see [the cy.fuel page](../docs/sources/cy-fuel.md).)

In [ ]:
%pip install -q "apographe[cy-fuel]"

In [ ]:
report = run("cy.fuel", out="data")
report.rows

In [ ]:
fuel = pd.read_csv(report.files[0])
fuel.head()

## 4. Collect from somewhere else

A source is a class with a name and one or more jobs. A job fetches data and writes tables through `out`. Here the data is invented; in a real source you would fetch it with `PoliteClient`.

In [ ]:
from apographe import Report, Source, run


class Stations(Source):
    id = "xx.fuel"  # country code, then a short name
    title = "Example fuel prices"
    country = "XX"
    homepage = "https://fuel.example.gov"
    jobs = {"prices": "every station's prices, now"}
    default_jobs = ("prices",)

    def run(self, job, out, options):
        rows = [
            {"station": "A", "price": 1.52},
            {"station": "B", "price": 1.49},
        ]
        path, count = out.write_csv(
            f"prices/{options.day}.csv", rows, ["station", "price"]
        )
        report = Report()
        report.add("prices", count, path)
        return report


report = run(Stations(), out="data")
report.files

In [ ]:
pd.read_csv(report.files[0])

Run it again tomorrow and you get a second dated file: that is how a history builds up.

Next: [using Apographe from Python](../docs/library.md), [adding a source](../docs/adding-a-source.md), [collecting every day](../docs/scheduling.md).